# Phase 4B — IPA-Whisper Base verified smoke test

**Revision: 2026-11-01-v6. Prepared notebook; not yet executed on Kaggle.**

Checks the pinned processor/checkpoint, all target strings, 12 training-only audio clips, generation termination, repeatability, inference memory/time and checkpoint save/reload. This does not fine-tune or score a validation/test baseline. Keep inputs, notebook and results private.

1. Import this file into Kaggle and attach the existing private Phase 4 smoke Dataset. ZIP and extracted bundles are supported; the first code cell lists mounted inputs.
2. Enable a GPU and Internet.
3. Run the install cell. If it changes installed packages, **restart the kernel/session before continuing**. The next cell checks an in-memory flag and imported package versions, stopping if a restart is still needed.
4. Run in order. After restart, the install cell skips reinstalling versions that already match. Do not change packages midway through the run.
5. Download `phase4_smoke_results_verified.zip` and the executed notebook after completion.

The model and processor revision is `949f7c4e4ee374b5202bc5886f6e4552e5ae2371`. The 42 symbol-review labels are tokenizer-only checks and remain excluded from the frozen pool. The 12 audio files are training-only; no validation/test audio is used.

Revision v6 corrects EOS/cap measurement using full short-form decoder sequences. The v5 measured run passed core engineering checks, but its stopping flags are invalid because its tensor output excluded prefix/EOS. This corrected generation route has not yet been run on Kaggle; verify decoded text against v5 on the same training clips before baseline scoring.


In [ ]:
# List every file Kaggle exposes from the datasets attached to this notebook.
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    raise FileNotFoundError(f'Kaggle input directory is not available: {INPUT_ROOT}')

input_files = sorted(path for path in INPUT_ROOT.rglob('*') if path.is_file())
if not input_files:
    print('No attached input files found under /kaggle/input.')
else:
    print(f'Found {len(input_files)} files under {INPUT_ROOT}:')
    for path in input_files:
        print(path)


## 1. Install packages, restart if changed, then verify imports

Keep Kaggle's CUDA PyTorch. Pin Transformers, its tokenizer/Hub dependencies, and SoundFile. The previous run had an imported Transformers version different from installed metadata; this notebook stops on that mismatch. Other preinstalled applications may still report unrelated dependency conflicts; the package report records `pip check` output.


In [ ]:
import subprocess
import sys
import json
import hashlib
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

EXPECTED_PACKAGES = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "soundfile": "0.13.1",
}

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

needs_install = any(installed_version(p) != v for p, v in EXPECTED_PACKAGES.items())
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        *[f"{p}=={v}" for p, v in EXPECTED_PACKAGES.items()],
    ])
    # This in-memory flag disappears when Python actually restarts.
    # Do not use connection/signing keys: Jupyter may reuse them.
    sys._phase4_packages_changed = True
    print("Packages installed. RESTART the kernel/session before continuing.")
else:
    print("Pinned packages are already installed; no installation performed.")


In [ ]:
from pathlib import Path
import hashlib
from copy import deepcopy
import json
from importlib.metadata import PackageNotFoundError, version
import platform
import re
import tempfile
import time
import unicodedata
import zipfile

# An in-memory flag is cleared by a Python kernel restart.
# Ignore old on-disk restart markers from earlier notebook revisions.
import sys
if getattr(sys, "_phase4_packages_changed", False):
    raise RuntimeError("Packages were installed in this Python process. Restart the kernel first.")

# Detect stale imported dependencies before loading the model stack.
for module_name, distribution_name in (
    ("transformers", "transformers"),
    ("tokenizers", "tokenizers"),
    ("huggingface_hub", "huggingface-hub"),
    ("soundfile", "soundfile"),
):
    loaded = sys.modules.get(module_name)
    if loaded is not None:
        loaded_version = getattr(loaded, "__version__", None)
        if loaded_version is not None and loaded_version != version(distribution_name):
            raise RuntimeError(
                f"{module_name}: imported {loaded_version}, installed {version(distribution_name)}. "
                "Restart the kernel before continuing."
            )

import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import display
from transformers import WhisperConfig, WhisperForConditionalGeneration, WhisperProcessor

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
PACKAGE_VERSIONS = {}
for package_name in ("torch", "transformers", "tokenizers", "huggingface-hub", "safetensors", "numpy", "soundfile"):
    try:
        PACKAGE_VERSIONS[package_name] = version(package_name)
    except PackageNotFoundError:
        PACKAGE_VERSIONS[package_name] = None
EXPECTED_PACKAGES = {
    "transformers": "4.57.1", "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2", "soundfile": "0.13.1",
}
for package, expected in EXPECTED_PACKAGES.items():
    if version(package) != expected:
        raise RuntimeError(f"{package}: expected {expected}, found {version(package)}. Run install and restart.")
if transformers.__version__ != version("transformers"):
    raise RuntimeError("Loaded Transformers differs from installed metadata. Restart the kernel.")
print("Transformers module path:", transformers.__file__)
import subprocess
import sys
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True
)
PIP_CHECK = {"returncode": pip_check.returncode, "stdout": pip_check.stdout, "stderr": pip_check.stderr}
print("pip check return code:", pip_check.returncode, "(recorded in report)")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No Kaggle CUDA GPU is active. Open notebook Settings → Accelerator → GPU, save, then restart and rerun.")

# Kaggle mounts attached private datasets under /kaggle/input.
INPUT_ROOT = Path("/kaggle/input")
required_files = {"bundle_record.json", "smoke_manifest.csv", "tokenizer_targets.csv"}

# Some uploads remain a ZIP; others may appear as already-extracted files.
usable_zips = []
for candidate_zip in INPUT_ROOT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(candidate_zip) as archive:
            names = {Path(name).name for name in archive.namelist() if not name.endswith("/")}
        if required_files.issubset(names):
            usable_zips.append(candidate_zip)
    except zipfile.BadZipFile:
        continue

direct_bundles = [
    record.parent for record in INPUT_ROOT.rglob("bundle_record.json")
    if all((record.parent / name).is_file() for name in ("smoke_manifest.csv", "tokenizer_targets.csv"))
]
if len(usable_zips) == 1:
    INPUT_DIR = Path("/kaggle/working/phase4_smoke_input")
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(usable_zips[0]) as archive:
        archive.extractall(INPUT_DIR)
    bundle_records = list(INPUT_DIR.rglob("bundle_record.json"))
    if len(bundle_records) != 1:
        raise RuntimeError(f"Expected one bundle_record.json after extraction; found {len(bundle_records)}.")
    INPUT_DIR = bundle_records[0].parent
elif len(usable_zips) == 0 and len(direct_bundles) == 1:
    INPUT_DIR = direct_bundles[0]
else:
    visible_files = [str(p.relative_to(INPUT_ROOT)) for p in INPUT_ROOT.rglob("*") if p.is_file()][:40]
    raise FileNotFoundError(
        "Could not identify exactly one Phase 4 bundle. In the Kaggle notebook, open the Input panel, "
        "choose Add Input, and attach your private Phase 4 dataset. "
        f"Found {len(usable_zips)} matching ZIPs and {len(direct_bundles)} extracted bundles. "
        f"Files currently visible under /kaggle/input: {visible_files}"
    )

AUDIO_DIR = INPUT_DIR / "audio"
SMOKE_CSV = INPUT_DIR / "smoke_manifest.csv"
TARGETS_CSV = INPUT_DIR / "tokenizer_targets.csv"
BUNDLE_RECORD = json.loads((INPUT_DIR / "bundle_record.json").read_text(encoding="utf-8"))
smoke = pd.read_csv(SMOKE_CSV, encoding="utf-8", keep_default_na=False)
targets = pd.read_csv(TARGETS_CSV, encoding="utf-8", keep_default_na=False)
# Check column names before later cells access them.
for name, table, required in (
    ("smoke", smoke, {"sample_id", "audio_file", "audio_sha256", "reference", "component_id"}),
    ("targets", targets, {"sample_id", "reference", "scope"}),
):
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f"{name} CSV is missing columns: {sorted(missing)}")
    if table["reference"].str.strip().eq("").any():
        raise ValueError(f"{name} CSV contains an empty reference.")

# All audio access uses this one helper. Manifest paths are relative to
# the bundle root (audio/ID.wav); bare filenames are also supported.
def resolve_audio_path(audio_file):
    relative = Path(str(audio_file))
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Expected a relative bundle audio path: {audio_file}")
    if relative.parts and relative.parts[0] == "audio":
        path = INPUT_DIR / relative
    else:
        path = AUDIO_DIR / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing audio: {path}; CSV value: {audio_file}")
    return path

if len(smoke) != 12 or smoke.sample_id.duplicated().any():
    raise ValueError("Expected exactly 12 unique smoke audio records.")
if len(targets) != 5086 or targets.sample_id.duplicated().any():
    raise ValueError("Expected 5,086 unique tokenizer target records.")
if targets.scope.value_counts().to_dict() != {"frozen_pool": 5044, "set_aside_symbol_review": 42}:
    raise ValueError("Tokenizer target scope counts do not match the frozen pool and review queue.")
if BUNDLE_RECORD.get("contains_validation_or_test_audio") is not False:
    raise ValueError("Bundle record does not confirm training-only smoke audio.")
if smoke.component_id.nunique() != 12:
    raise ValueError("Smoke clips must come from 12 distinct connected components.")
for row in smoke.itertuples(index=False):
    audio_path = resolve_audio_path(row.audio_file)
    actual_hash = hashlib.sha256(audio_path.read_bytes()).hexdigest()
    if actual_hash != row.audio_sha256:
        raise ValueError(f"Audio hash mismatch for {row.sample_id}.")
print("Attached bundle folder:", INPUT_DIR)
print("Training audio clips:", len(smoke), "| tokenizer target rows:", len(targets))

OUTPUT_DIR = Path("/kaggle/working/phase4_smoke_results_verified")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_HASHES = {
    name: hashlib.sha256((INPUT_DIR / name).read_bytes()).hexdigest()
    for name in ("bundle_record.json", "smoke_manifest.csv", "tokenizer_targets.csv")
}
SEED = 0
import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## 2. Load the exact processor and check every target string

The processor converts audio to log-Mel features and text to/from model tokens. A tokenizer round trip tests whether the text can be represented; it does not prove speech recognition quality. This cell checks all 5,044 frozen references and reports the 42 set-aside labels separately.


In [ ]:
MODEL_ID = "neurlang/ipa-whisper-base"
MODEL_REVISION = "949f7c4e4ee374b5202bc5886f6e4552e5ae2371"
processor = WhisperProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
config = WhisperConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer = processor.tokenizer

# Match the project's declared text convention while retaining every non-space mark.
def normalize_text(value):
    text = unicodedata.normalize("NFD", str(value)).strip()
    return re.sub(r" +", " ", text)

roundtrip_rows = []
unknown_id = tokenizer.unk_token_id
for row in targets.itertuples(index=False):
    reference = normalize_text(row.reference)
    encoded = tokenizer(reference, add_special_tokens=False, truncation=False)["input_ids"]
    decoded = normalize_text(tokenizer.decode(
        encoded, skip_special_tokens=True, clean_up_tokenization_spaces=False
    ))
    roundtrip_rows.append({
        "sample_id": row.sample_id,
        "scope": row.scope,
        "token_count": len(encoded),
        "unknown_token": bool(unknown_id is not None and unknown_id in encoded),
        "roundtrip_match": decoded == reference,
        "over_target_limit": len(tokenizer(reference, add_special_tokens=True, truncation=False)["input_ids"]) > int(config.max_target_positions),
    })

roundtrip = pd.DataFrame(roundtrip_rows)
roundtrip.to_csv(OUTPUT_DIR / "tokenizer_checks.csv", index=False, encoding="utf-8")
summary = roundtrip.groupby("scope").agg(
    rows=("sample_id", "size"),
    roundtrip_failures=("roundtrip_match", lambda values: int((~values).sum())),
    unknown_token_rows=("unknown_token", "sum"),
    over_target_limit=("over_target_limit", "sum"),
    max_content_tokens=("token_count", "max"),
)
display(summary)
print("Whisper max target positions:", config.max_target_positions)

frozen_check = roundtrip.loc[roundtrip.scope.eq("frozen_pool")]
review_check = roundtrip.loc[roundtrip.scope.eq("set_aside_symbol_review")]
frozen_bad = (
    ~frozen_check.roundtrip_match
    | frozen_check.unknown_token
    | frozen_check.over_target_limit
)
review_bad = (
    ~review_check.roundtrip_match
    | review_check.unknown_token
    | review_check.over_target_limit
)
if frozen_bad.any():
    display(frozen_check.loc[frozen_bad].head(20))
    raise RuntimeError("A frozen target failed tokenizer round-trip/length checks. Stop and resolve before baseline scoring.")
if review_bad.any():
    display(review_check.loc[review_bad].head(20))
    print("REVIEW: at least one set-aside label did not pass; it remains excluded from the frozen pool.")
else:
    print("All 42 set-aside labels are representable and fit the target-length check.")


## 3. Confirm GPU and load the pinned model

GPU is appropriate for model generation and memory measurement. Tokenizer checks above run on CPU and do not need a GPU. We use FP32 and batch size 1 for this first test to match the declared zero-shot protocol.


In [ ]:
DEVICE = torch.device("cuda:0")
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_safetensors=True,
    torch_dtype=torch.float32,
)

# Store inference settings in GenerationConfig. Do not mutate model.config:
# save_pretrained can migrate those legacy fields and change reload behavior.
# Keep checkpoint generation-length defaults; make IPA decoding settings explicit.
GENERATION_CONFIG = deepcopy(model.generation_config)
GENERATION_CONFIG.forced_decoder_ids = None
GENERATION_CONFIG.suppress_tokens = []
GENERATION_CONFIG.language = None
GENERATION_CONFIG.task = None
GENERATION_CONFIG.do_sample = False
GENERATION_CONFIG.num_beams = 1
GENERATION_CONFIG._from_model_config = False
model.generation_config = deepcopy(GENERATION_CONFIG)
model.eval().to(DEVICE)

resolved_generation = model.generation_config.to_dict()
print("Device:", DEVICE)
print("Parameter dtype:", next(model.parameters()).dtype)
print("Target positions:", model.config.max_target_positions)
print("Attention implementation:", getattr(model.config, "_attn_implementation", "library default"))
print("Generation: greedy (do_sample=False), num_beams=1; no forced language/task; suppress_tokens=[]; inherited begin_suppress_tokens recorded")
print("Resolved max_length:", resolved_generation.get("max_length"))
print("Resolved max_new_tokens:", resolved_generation.get("max_new_tokens"))

print("begin_suppress_tokens:", GENERATION_CONFIG.begin_suppress_tokens)


## 4. Check audio loading and Whisper feature creation

Every audio clip should be non-empty, mono, and sampled at 16 kHz. The processor should return finite input features with Whisper's expected 80 Mel bins. This notebook does not trim, denoise, resample or normalize the source audio manually.


In [ ]:
def read_smoke_audio(row):
    audio_path = resolve_audio_path(row.audio_file)
    waveform, sample_rate = sf.read(audio_path, dtype="float32", always_2d=True)
    if sample_rate != 16000 or waveform.shape[1] != 1 or waveform.shape[0] == 0:
        raise ValueError(f"Unexpected audio format for {row.sample_id}: sr={sample_rate}, shape={waveform.shape}")
    duration = waveform.shape[0] / sample_rate
    if not 0.5 <= duration <= 30.0:
        raise ValueError(f"Out-of-window audio for {row.sample_id}: {duration:.6f}s")
    if not torch.isfinite(torch.from_numpy(waveform)).all():
        raise ValueError(f"Non-finite waveform for {row.sample_id}")
    return waveform[:, 0], sample_rate

feature_records = []
for row in smoke.itertuples(index=False):
    waveform, sample_rate = read_smoke_audio(row)
    features = processor(waveform, sampling_rate=sample_rate, return_tensors="pt").input_features
    if features.ndim != 3 or features.shape[1] != 80 or not torch.isfinite(features).all():
        raise ValueError(f"Invalid Whisper features for {row.sample_id}: shape={tuple(features.shape)}")
    feature_records.append({
        "sample_id": row.sample_id,
        "duration_seconds": len(waveform) / sample_rate,
        "feature_shape": list(features.shape),
        "finite_features": bool(torch.isfinite(features).all()),
    })
display(pd.DataFrame(feature_records))
print("Audio/feature checks passed for all 12 training clips.")

pd.DataFrame(feature_records).to_csv(OUTPUT_DIR / "feature_checks.csv", index=False)


## 5. Generation, stopping diagnostics and memory/time

Keep greedy decoding and checkpoint generation length defaults. Save raw token IDs and hypotheses, EOS/length-limit flags, replacement-character counts, and the longest consecutive run of a single token. This last measure detects some repetition but is not a linguistic accuracy metric. A cap hit does not by itself prove truncation; record EOS separately. Preserve all predictions.

The first clip is repeated to check token equality. Per-clip times exclude that repetition; loop wall time includes it, audio loading and processing. Memory figures are PyTorch allocator measurements, not total GPU occupancy or training requirements.


In [ ]:
def infer_tokens(inference_model, inference_processor, waveform, sample_rate, generation_config):
    features = inference_processor(
        waveform, sampling_rate=sample_rate, return_tensors="pt"
    ).input_features.to(DEVICE)
    with torch.inference_mode():
        ids = inference_model.generate(
            input_features=features,
            generation_config=deepcopy(generation_config),
            return_dict_in_generate=True,
        )
    # Whisper's default tensor strips prefix/EOS. The short-form
    # ModelOutput path retains full sequences for termination diagnostics.
    return ids.sequences[0].detach().cpu()


def generate_once(waveform, sample_rate):
    return infer_tokens(model, processor, waveform, sample_rate, GENERATION_CONFIG)


def longest_identical_token_run(token_ids):
    longest = current = 0
    previous = None
    for token_id in token_ids:
        current = current + 1 if token_id == previous else 1
        longest = max(longest, current)
        previous = token_id
    return longest


if GENERATION_CONFIG.max_new_tokens is not None:
    raise RuntimeError("Expected checkpoint length defaults with max_new_tokens=None.")
eos_value = GENERATION_CONFIG.eos_token_id
if eos_value is None:
    raise RuntimeError("Generation configuration has no EOS token.")
eos_ids = {int(eos_value)} if isinstance(eos_value, int) else set(eos_value)

model.eval()
prediction_rows = []
token_records = []
repeatability = False
torch.cuda.synchronize(DEVICE)
torch.cuda.reset_peak_memory_stats(DEVICE)
start_all = time.perf_counter()
for i, row in enumerate(smoke.itertuples(index=False)):
    waveform, sample_rate = read_smoke_audio(row)
    torch.cuda.synchronize(DEVICE)
    start = time.perf_counter()
    ids = generate_once(waveform, sample_rate)
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - start
    if i == 0:
        repeated_ids = generate_once(waveform, sample_rate)
        repeatability = bool(torch.equal(ids, repeated_ids))
        if not repeatability:
            raise RuntimeError("Repeated first-clip greedy generation differed.")
    token_ids = ids.tolist()
    hypothesis = processor.tokenizer.decode(
        token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    ended_with_eos = bool(token_ids and token_ids[-1] in eos_ids)
    cap_hit = len(token_ids) >= int(GENERATION_CONFIG.max_length)
    prediction_rows.append({
        "sample_id": row.sample_id,
        "reference": row.reference,
        "prediction": hypothesis,
        "inference_seconds": elapsed,
        "duration_seconds": len(waveform) / sample_rate,
        "returned_token_count_including_prefix": len(token_ids),
        "eos_after_start": any(t in eos_ids for t in token_ids[1:]),
        "ended_with_eos": ended_with_eos,
        "reached_length_limit": cap_hit,
        "possible_truncation": cap_hit and not ended_with_eos,
        "replacement_character_count": hypothesis.count("\ufffd"),
        "longest_identical_token_run": longest_identical_token_run(token_ids),
    })
    token_records.append({"sample_id": row.sample_id, "token_ids": token_ids})
    # Save incrementally so later failures do not erase earlier diagnostics.
    pd.DataFrame(prediction_rows).to_csv(
        OUTPUT_DIR / "smoke_predictions_training_only.csv", index=False, encoding="utf-8"
    )
    (OUTPUT_DIR / "generated_token_ids.json").write_text(
        json.dumps(token_records, indent=2), encoding="utf-8"
    )
torch.cuda.synchronize(DEVICE)
total_generation_seconds = time.perf_counter() - start_all
peak_allocated_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
peak_reserved_mb = torch.cuda.max_memory_reserved(DEVICE) / (1024 ** 2)
predictions = pd.DataFrame(prediction_rows)
display(predictions)
print("Repeated generation identical:", repeatability)
print("Generation-loop wall seconds (includes incremental exports):", total_generation_seconds)
print("Peak allocated/reserved GPU MiB:", peak_allocated_mb, peak_reserved_mb)
print("Length-limit hits:", int(predictions.reached_length_limit.sum()))
print("Predictions containing replacement characters:", int(predictions.replacement_character_count.gt(0).sum()))


## 6. Save and reload the checkpoint, then compare output

This writes the pinned model and processor to a temporary Kaggle directory, reloads them, and verifies that the first clip produces the same token IDs. The temporary checkpoint is deleted automatically after the test; the published base checkpoint remains on Hugging Face.


In [ ]:
from transformers import GenerationConfig

# Keep the working model alive: a reload failure must not destroy it.
checkpoint_roundtrip_identical = False
first_row = next(smoke.itertuples(index=False))
first_waveform, first_sample_rate = read_smoke_audio(first_row)
original_ids = generate_once(first_waveform, first_sample_rate)
reloaded_model = None

try:
    with tempfile.TemporaryDirectory(prefix="phase4_whisper_roundtrip_") as checkpoint_dir:
        # Persist the same explicit configuration used for original generation.
        model.generation_config = deepcopy(GENERATION_CONFIG)
        model.save_pretrained(checkpoint_dir, safe_serialization=True)
        processor.save_pretrained(checkpoint_dir)
        GENERATION_CONFIG.save_pretrained(checkpoint_dir)

        saved_generation = GenerationConfig.from_pretrained(checkpoint_dir)
        # Ignore provenance bookkeeping when comparing settings.
        def generation_settings(config):
            return {k: v for k, v in config.to_dict().items()
                    if k not in {"transformers_version", "_commit_hash", "_from_model_config"}}

        if generation_settings(saved_generation) != generation_settings(GENERATION_CONFIG):
            raise RuntimeError("Saved generation settings differ from the inference settings.")

        reloaded_processor = WhisperProcessor.from_pretrained(checkpoint_dir)
        reloaded_model = WhisperForConditionalGeneration.from_pretrained(
            checkpoint_dir, use_safetensors=True, torch_dtype=torch.float32
        ).to(DEVICE)
        reloaded_model.eval()
        reloaded_ids = infer_tokens(
            reloaded_model, reloaded_processor,
            first_waveform, first_sample_rate, saved_generation,
        )
        checkpoint_roundtrip_identical = bool(torch.equal(original_ids, reloaded_ids))
        (OUTPUT_DIR / "checkpoint_roundtrip.json").write_text(json.dumps({
            "sample_id": first_row.sample_id,
            "original_ids": original_ids.tolist(),
            "reloaded_ids": reloaded_ids.tolist(),
            "identical": checkpoint_roundtrip_identical,
        }, indent=2), encoding="utf-8")
        print("Checkpoint save/reload output identical:", checkpoint_roundtrip_identical)
        if not checkpoint_roundtrip_identical:
            print("Original IDs:", original_ids.tolist())
            print("Reloaded IDs:", reloaded_ids.tolist())
            raise RuntimeError(
                "Token IDs differ after reload with matched settings. "
                "The original model is retained; stop and investigate before baseline evaluation."
            )
finally:
    # Release only the extra model, even if the check raises an exception.
    if reloaded_model is not None:
        del reloaded_model
    torch.cuda.empty_cache()


## 7. Visual results for the smoke test

These five figures use this run's saved measurements: clip duration/latency, reference and generation token lengths, text growth/repetition, stopping/Unicode flags, and GPU memory. All audio examples belong to training; the reference histogram includes the tokenizer-only frozen and set-aside scopes. The rare 42-row scope may be hard to see next to 5,044 rows; both are included in the histogram with a log count axis.

The reference histogram excludes special tokens; the earlier eligibility check includes them. Generated lengths include the decoder prefix. Length differences are not CER, and repetition/termination flags can overlap. The first clip's latency can include warm-up effects. Figures and their source table remain private and are included in the downloadable results ZIP.


In [ ]:
# Visual summaries for this run's 12 training-only smoke examples.
# These are engineering diagnostics, not validation/test accuracy plots.
import matplotlib
import matplotlib.pyplot as plt

PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
plot_files = []


def save_figure(fig, filename):
    path = PLOT_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plot_files.append(path)
    plt.show()
    plt.close(fig)


plot_data = predictions.copy().sort_values("duration_seconds").reset_index(drop=True)
plot_data["reference_nfd_characters"] = plot_data["reference"].map(
    lambda text: len(normalize_text(text))
)
plot_data["prediction_nfd_characters"] = plot_data["prediction"].map(
    lambda text: len(normalize_text(text))
)
labels = plot_data["sample_id"].tolist()
x = np.arange(len(plot_data))

with plt.rc_context({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False}):
    # 1. Input duration and measured end-to-end single-clip latency.
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
    axes[0].bar(x, plot_data["duration_seconds"], color="#377eb8")
    axes[0].set(title="Smoke audio durations", ylabel="Audio seconds")
    axes[0].set_xticks(x, labels, rotation=65, ha="right")
    axes[1].scatter(plot_data["duration_seconds"], plot_data["inference_seconds"],
                    s=55, color="#377eb8")
    for _, row in plot_data.iterrows():
        axes[1].annotate(row["sample_id"],
                         (row["duration_seconds"], row["inference_seconds"]),
                         xytext=(3, 5), textcoords="offset points", fontsize=7)
    axes[1].set(title="Audio duration vs inference time", xlabel="Audio seconds",
                ylabel="Inference seconds (features + generation)")
    fig.suptitle("Training-only smoke test: inputs and timing")
    save_figure(fig, "01_inputs_and_timing.png")

    # 2. Label capacity and model output lengths are different checks.
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
    for scope, group in roundtrip.groupby("scope"):
        axes[0].hist(group["token_count"], bins=np.arange(0, 481, 20),
                     alpha=0.65, label=f"{scope} (n={len(group):,})")
    axes[0].axvline(config.max_target_positions, color="#e41a1c", linestyle="--",
                    label=f"Decoder capacity: {config.max_target_positions}")
    axes[0].set(title="Reference content-token lengths", xlabel="Content tokens (prefix/EOS excluded)",
                ylabel="Reference count", yscale="log")
    axes[0].legend(fontsize=8)
    axes[1].bar(x, plot_data["returned_token_count_including_prefix"], color="#984ea3")
    axes[1].axhline(GENERATION_CONFIG.max_length, color="#e41a1c", linestyle="--",
                    label=f"Generation limit: {GENERATION_CONFIG.max_length}")
    axes[1].set(title="Generated sequence lengths", ylabel="Returned tokens (prefix included)")
    axes[1].set_xticks(x, labels, rotation=65, ha="right")
    axes[1].legend(fontsize=8)
    fig.suptitle("Reference representation vs full generation length (different denominators)")
    save_figure(fig, "02_token_lengths.png")

    # 3. Output growth does not measure transcription accuracy.
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
    axes[0].bar(x - 0.2, plot_data["reference_nfd_characters"], width=0.4,
                label="Reference", color="#377eb8")
    axes[0].bar(x + 0.2, plot_data["prediction_nfd_characters"], width=0.4,
                label="Prediction", color="#ff7f00")
    axes[0].set(title="Normalized text lengths (not CER)", ylabel="NFD code points, including spaces")
    axes[0].set_xticks(x, labels, rotation=65, ha="right")
    axes[0].legend()
    axes[1].bar(x, plot_data["longest_identical_token_run"], color="#984ea3")
    axes[1].set(title="Consecutive identical-token repetition", ylabel="Longest run in returned token IDs")
    axes[1].set_xticks(x, labels, rotation=65, ha="right")
    fig.suptitle("Training-only output diagnostics: growth and repetition")
    save_figure(fig, "03_output_growth_and_repetition.png")

    # 4. Indicators overlap: a cap hit can also end with EOS.
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    flag_columns = ["ended_with_eos", "reached_length_limit", "possible_truncation"]
    flags = plot_data[flag_columns].astype(int).copy()
    flags["replacement_character_present"] = plot_data["replacement_character_count"].gt(0).astype(int)
    ax.imshow(flags.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    ax.set_xticks(np.arange(4), ["Ended with EOS", "Reached length limit", "Cap without EOS", "Contains U+FFFD"],
                   rotation=15, ha="right")
    ax.set_yticks(x, labels)
    for i in range(len(flags)):
        for j in range(4):
            value = int(flags.iloc[i, j])
            ax.text(j, i, "Yes" if value else "No", ha="center", va="center",
                    color="white" if value else "black", fontsize=9)
    ax.set_title("Generation stopping and Unicode flags — all 12 smoke clips")
    save_figure(fig, "04_stopping_and_unicode_flags.png")

    # 5. The counters cover the generation loop, not save/reload or training.
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    bars = ax.bar(["Peak allocated", "Peak reserved"],
                  [peak_allocated_mb, peak_reserved_mb], color=["#377eb8", "#984ea3"])
    ax.bar_label(bars, fmt="%.1f MiB", padding=4)
    ax.set_ylim(0, max(peak_allocated_mb, peak_reserved_mb, 1) * 1.2)
    ax.set(ylabel="PyTorch CUDA memory (MiB)",
           title="Batch-1 FP32 inference memory (not training memory)")
    save_figure(fig, "05_inference_memory.png")

plot_data.to_csv(OUTPUT_DIR / "plot_data_training_only.csv", index=False, encoding="utf-8")
print(f"Saved {len(plot_files)} figures to {PLOT_DIR}")
print("Repetition metric detects identical-token runs only; repeated multi-token phrases may be missed.")
print("Text length differences and token counts are diagnostics, not accuracy scores.")


## 8. Export the engineering results and diagnostics

The ZIP contains environment and dependency-conflict records, input hashes, tokenizer/feature rows, raw generation tokens, predictions, stopping flags and reload token comparison. `engineering_checks_passed` does not assert good Nyishi accuracy. Repetition, replacement characters and cap hits remain in the saved data. No CER is computed and no validation/test audio is accessed.


In [ ]:
# Refresh environment metadata at export; reject mixed import/install versions.
PACKAGE_VERSIONS = {
    name: version(name) for name in (
        "torch", "transformers", "tokenizers", "huggingface-hub",
        "safetensors", "numpy", "soundfile", "matplotlib",
    )
}
if transformers.__version__ != PACKAGE_VERSIONS["transformers"]:
    raise RuntimeError("Transformers module/metadata mismatch at export. Restart and rerun.")

checks = {
    "environment_matches_pin": transformers.__version__ == "4.57.1",
    "frozen_tokenizer_passed": not bool(frozen_bad.any()),
    "feature_checks_passed": len(feature_records) == 12,
    "all_training_examples_generated": len(predictions) == 12,
    "repeat_generation_identical": bool(repeatability),
    "checkpoint_save_reload_identical": bool(checkpoint_roundtrip_identical),
}
report = {
    "phase": "4_processor_model_smoke_test",
    "notebook_revision": "2026-11-01-v6",
    "status": "engineering_checks_passed" if all(checks.values()) else "failed",
    "checks": checks,
    "plot_files": [str(path.relative_to(OUTPUT_DIR)) for path in plot_files],
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "processor_revision": MODEL_REVISION,
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "transformers_module_path": transformers.__file__,
    "package_versions": PACKAGE_VERSIONS,
    "pip_check": PIP_CHECK,
    "seed": SEED,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0),
    "cuda_version": torch.version.cuda,
    "precision": "float32",
    "attention_implementation": getattr(model.config, "_attn_implementation", None),
    "input_file_sha256": INPUT_HASHES,
    "bundle_record": BUNDLE_RECORD,
    "frozen_tokenizer_targets": int(len(frozen_check)),
    "frozen_roundtrip_failures": int((~frozen_check.roundtrip_match).sum()),
    "frozen_unknown_token_rows": int(frozen_check.unknown_token.sum()),
    "frozen_over_target_limit": int(frozen_check.over_target_limit.sum()),
    "set_aside_symbol_targets": int(len(review_check)),
    "set_aside_failed_checks": int(review_bad.sum()),
    "smoke_audio_scope": "training only",
    "smoke_audio_count": int(len(predictions)),
    "smoke_audio_seconds": float(predictions.duration_seconds.sum()),
    "generation_wall_seconds": float(total_generation_seconds),
    "timing_note": "Loop includes repeated first inference, audio/features and incremental exports; excludes model loading and save/reload.",
    "peak_allocated_gpu_mib": float(peak_allocated_mb),
    "peak_reserved_gpu_mib": float(peak_reserved_mb),
    "length_limit_hits": int(predictions.reached_length_limit.sum()),
    "possible_truncations": int(predictions.possible_truncation.sum()),
    "predictions_ending_with_eos": int(predictions.ended_with_eos.sum()),
    "predictions_with_replacement_characters": int(predictions.replacement_character_count.gt(0).sum()),
    "repeat_generation_identical": bool(repeatability),
    "checkpoint_save_reload_identical": bool(checkpoint_roundtrip_identical),
    "cer_computed": False,
    "accuracy_evaluation": "Not performed. These are training-only engineering examples; inspect output quality before baseline scoring.",
}
(OUTPUT_DIR / "smoke_summary.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
(OUTPUT_DIR / "generation_config.json").write_text(
    json.dumps(resolved_generation, indent=2, ensure_ascii=False), encoding="utf-8"
)
results_zip = Path("/kaggle/working/phase4_smoke_results_verified.zip")
# Include only this notebook's named outputs, not unrelated files in working/.
artifact_names = [
    "smoke_summary.json", "generation_config.json", "smoke_predictions_training_only.csv",
    "generated_token_ids.json", "tokenizer_checks.csv", "feature_checks.csv", "checkpoint_roundtrip.json",
    "plot_data_training_only.csv",
]
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in artifact_names:
        path = OUTPUT_DIR / name
        if not path.is_file():
            raise FileNotFoundError(f"Missing run artifact: {path}")
        archive.write(path, arcname=name)
    for path in plot_files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing figure: {path}")
        archive.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Download private results:", results_zip)


## Interpreting the result

- A restart warning means packages changed inside this kernel: restart before importing model libraries. Do not bypass the imported/installed version check.
- The first cell lists files already attached to Kaggle; it does not download data. Dataset discovery accepts extracted files or a ZIP.
- A label check failure requires investigation; preserve all IPA marks.
- A generation cap hit or repetitive output is recorded as a model limitation. Do not drop these examples or strip replacement characters to improve a score.
- The inherited `begin_suppress_tokens` remains recorded alongside `suppress_tokens=[]`. No language or task is explicitly forced.
- Reload compares token IDs for one clip on this runtime and retains the original model if it fails. This is not cross-hardware reproducibility evidence.
- Once engineering gates pass and output behavior is understood, proceed to frozen **validation** inference with the declared NFD code-point corpus CER policy. Test remains reserved for final evaluation.
